[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/introduction/25_Seed_Data.ipynb)

> **中文译文** —— 由社区翻译，非上游原文；源文件变更后以英文原版为准。步骤间的逻辑导读为译者补充。

# 种子数据（Seed Data）—— 实战指南

`seed` 模块用**可信的已知数据**（CSV/JSON/数据库/API 来源）在抽取运行*之前*为知识图谱打底。抽取因此有了可挂靠的地基，而不是从一张空图起步。

关键部件：

- **`SeedDataManager`** - 注册数据源、构建基础图
- **`create_foundation_graph()`** - 把注册来源变成 `entities` + `relationships` + `metadata`
- **`validate_quality()`** - 在你提交基础图之前做质量检查

以下示例均对照 `semantica/seed/seed_manager.py` 实际执行。

### 先看主线

为什么需要种子？05/06 篇的抽取从零开始——实体没有锚点，全靠文本自己长。种子数据反过来：先把*已知的、可信的*实体放进图里，抽取结果往地基上挂，图的起点不再是空白。四节两段——**备料 → 成图**：

**备料（第 1–2 节）—— 登记与加载**

1. 第 1 节 注册来源：可信数据住在哪里、什么格式、什么实体类型
2. 第 2 节 加载记录：读入并给每条记录附上 `entity_type` 和 `source` 溯源键——22 篇的溯源纪律从种子就开始

**成图（第 3–4 节）—— 建图与把关**

3. 第 3 节 建基础图：来源转成图就绪的实体与关系，`confidence: 1.0`——种子数据的可信是定义出来的，不是算出来的
4. 第 4 节 质量把关：`validate_quality` 在提交前拦下坏种子——脏数据进图就是污染全图

In [1]:
!pip install -q semantica

## 1) 准备种子 CSV 并注册来源

`register_source(name, format, location, entity_type=...)` 记录可信数据的位置。`verified=True`（默认）标记来源已预先验证。

In [2]:
import csv
import tempfile
from pathlib import Path
from semantica.seed import SeedDataManager

# Write the sample CSV into a session-scoped temp directory so we never
# clobber a companies.csv that might exist in the user's working directory.
seed_csv = Path(tempfile.mkdtemp(prefix="semantica-seed-")) / "companies.csv"
with open(seed_csv, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "name", "type", "industry"])
    writer.writeheader()
    writer.writerow({"id": "c1", "name": "Acme", "type": "Company", "industry": "robotics"})
    writer.writerow({"id": "c2", "name": "Globex", "type": "Company", "industry": "energy"})

manager = SeedDataManager()
manager.register_source("companies", format="csv", location=str(seed_csv), entity_type="Company")


True

## 2) 从注册来源加载记录

`load_source(name)` 读取来源，并为每条记录附上 `entity_type` 和 `source` 溯源键。

In [3]:
records = manager.load_source("companies")
print(f"loaded {len(records)} records")
records[0]

Status,Action,Module,Submodule,Progress,ETA,Rate,Time,Extracted
✅,Semantica is seeding,🌱 seed,SeedDataManager,100.0%,-,-,0.00s,-


🔄 Semantica is seeding: Loading seed data from CSV: /var/folders/7s/bvvstgs10y963tz6_4bbnklr0000gn/T/semantica-seed-eu9__ep1/companies.csv 🌱 seed SeedDataManager |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.00s Extracted: -

loaded 2 records


{'id': 'c1',
 'name': 'Acme',
 'type': 'Company',
 'industry': 'robotics',
 'entity_type': 'Company',
 'source': 'companies'}

## 3) 构建基础图

`create_foundation_graph()` 把每个注册来源转换成图就绪的实体和关系。实体带 `confidence: 1.0`——种子数据定义上可信。

In [4]:
foundation = manager.create_foundation_graph()
sorted(foundation.keys())

['entities', 'metadata', 'relationships']

In [5]:
foundation["entities"][0]

{'id': 'c1',
 'text': 'Acme',
 'type': 'Company',
 'confidence': 1.0,
 'metadata': {'industry': 'robotics', 'source': 'companies'}}

## 4) 提交前验证质量

`validate_quality(foundation_graph)` 返回 `valid`、`errors`、`warnings` 和 `metrics`——坏种子在污染图谱之前就被拦在闸门外。

In [6]:
quality = manager.validate_quality(foundation)
quality["valid"]

True

## 小结

### 回顾主线

备料与成图走完：**备料**（注册可信来源、带溯源键地加载）、**成图**（confidence 1.0 的基础图、质量闸门把关）。地基打好，05/06 篇的抽取结果就有了挂靠之处——`integrate_with_extracted` 负责两路汇合。

| 任务 | API |
|---|---|
| 注册可信来源 | `register_source(name, format, location, entity_type=...)` |
| 加载记录 | `load_source(name)` —— 附 `entity_type` / `source` 键 |
| 直接读文件 | `load_from_csv(path)` / `load_from_json(path)` |
| 构建图 | `create_foundation_graph()` → `entities` / `relationships` / `metadata` |
| 拦住坏数据 | `validate_quality(graph)` → `valid` / `errors` / `warnings` / `metrics` |

另见 `semantica/seed/seed_usage.md`：`load_from_database`、`load_from_api` 和 `integrate_with_extracted`。